# 04m — Balanceamento do treino multiclasse (RF e SVM)

Este notebook **não treina nada**. Ele monta, a partir da view definitiva do notebook 03m, o **conjunto de treino** do multiclasse (as 7 classes, no máximo **50 mil linhas reais por classe**) e uma **validação separada**, para o RF e o SVM multiclasse.

## Decisões (e por quê)

| Pergunta | Decisão |
|---|---|
| **Quantas linhas por classe?** | Até **50.000 linhas reais por classe** (`TETO_POR_CLASSE`). Classes com menos linhas usam **todas** (por exemplo, `NetBIOS` e `LDAP`, que têm menos de 50 mil no treino). **Nenhuma linha sintética** (sem SMOTE) |
| **Como compensar as classes menores?** | Com `class_weight="balanced"` **no treino do modelo** (notebook 05m), calculado a partir do ajuste. Este notebook só mostra os pesos que resultariam |
| **Como dividir?** | **80% de cada classe para o ajuste** e 20% para a validação (`FRACAO_AJUSTE`), por sorteio pseudoaleatório fixo dentro da classe. A validação só tem linhas **reais** e nunca se mistura com o ajuste |
| **Quem recebe esse conjunto?** | **O mesmo** para RF e SVM, para a comparação ser justa |
| **E o teste?** | **Não é balanceado**: continua sendo `prep_vw_rf_multi_teste_sem_sobrep`, como está, com a proporção real das classes |
| **Pesos da validação** | Cada linha da validação tem o peso `linhas da classe no treino ÷ linhas da classe na validação`: com eles, a validação ponderada tem as proporções do treino das 7 classes (o análogo do que o binário fazia). **A decisão de modelos usa a validação sem pesos** (as classes ficam equilibradas); a ponderada serve para relatar métricas que dependem da proporção |

Por que sem SMOTE e com `class_weight`: as classes menores têm 20 a 30 mil linhas contra 50 mil, um desequilíbrio leve (2,6 para 1 no máximo). Com 50 mil por classe, o SMOTE precisaria inventar 40% (`LDAP`) e 61% (`NetBIOS`) das linhas. O peso por classe compensa o desequilíbrio sem inventar dados e sem risco de vazamento. O SMOTE-NC fica como possível variante futura.

## O que é criado (nada existente é alterado)

| Objeto | Conteúdo |
|---|---|
| `prep_bal_multi_selecao` | Quais linhas foram escolhidas, com o papel (`treino` ou `validacao`), a classe (`tipo`), o código `y` e o peso. É a **auditoria** da seleção |
| `prep_bal_multi_dados` | Os dados dessas linhas, copiados da view definitiva (nulos já pela mediana). Tabela pequena: leitura rápida e seleção congelada |
| `prep_vw_bal_multi_treino` | `id_linha`, `y` (código da classe) e atributos do conjunto de ajuste |
| `prep_vw_bal_multi_validacao` | Igual, mais a coluna `peso` |

A classe por nome (`tipo`) fica só na tabela de auditoria e **nunca** nas views de treino: não pode ser atributo do modelo.

## Antes de rodar
O notebook 03m precisa ter criado `prep_vw_rf_multi_treino` e `prep_classes_rf_multi`. A conexão é a mesma do 02 e do 03m (arquivo `.env`): coloque este arquivo na **mesma pasta**.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas ----------
import math
import numpy as np
from datetime import datetime

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte. Aqui, se o servidor estiver sem memória (Code 241) ou a conexão cair, esperamos e refazemos
    # (o comando `antes`, por exemplo um DROP, devolve o estado inicial; tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

## 1. Parâmetros e o que existe

- `TETO_POR_CLASSE`: máximo de linhas **reais** de cada classe (ajuste + validação). Classes com menos linhas usam todas.
- `FRACAO_AJUSTE`: fração de cada classe que vai para o **ajuste** do modelo. O restante vira validação.
- `SEMENTE`: fixa; a mesma seleção sai sempre. **Não a mude depois de ver resultados.**

A tabela abaixo mostra, para cada classe, quantas linhas existem no treino (já sem duplicatas), quantas serão usadas e como se dividem.

In [5]:
# ---------- 1.1 Parâmetros e levantamento ----------
SEMENTE = globals().get("SEMENTE", 20261018)
TETO_POR_CLASSE = globals().get("TETO_POR_CLASSE", 50_000)
FRACAO_AJUSTE = globals().get("FRACAO_AJUSTE", 0.80)
VIEW_BASE, VIEW_TESTE, TAB_CLASSES = "prep_vw_rf_multi_treino", "prep_vw_rf_multi_teste_sem_sobrep", "prep_classes_rf_multi"       # views e auditoria do notebook 03m
SEL_TAB, DADOS_TAB, V_TREINO, V_VALID = "prep_bal_multi_selecao", "prep_bal_multi_dados", "prep_vw_bal_multi_treino", "prep_vw_bal_multi_validacao"

faltam = [t for t in ("prep_final_treino", VIEW_BASE, VIEW_TESTE, TAB_CLASSES) if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02 e 03m: faltam {faltam}"
assert 0.5 <= FRACAO_AJUSTE < 1.0, "FRACAO_AJUSTE deve ficar entre 0,5 e 1 (precisa sobrar linhas para a validação)"
assert TETO_POR_CLASSE >= 100, "TETO_POR_CLASSE muito pequeno"

cl = consultar_num(f"SELECT codigo, classe, linhas_treino, linhas_teste_sem_sobrep FROM {TAB_CLASSES} ORDER BY codigo")
CLASSES = cl.classe.tolist(); COD = {c: int(k) for c, k in zip(cl.classe, cl.codigo)}; N_TREINO = {c: int(n) for c, n in zip(cl.classe, cl.linhas_treino)}; N_TESTE = {c: int(n) for c, n in zip(cl.classe, cl.linhas_teste_sem_sobrep)}
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
USAR = {c: min(int(TETO_POR_CLASSE), N_TREINO[c]) for c in CLASSES}; N_AJ = {c: int(FRACAO_AJUSTE * USAR[c]) for c in CLASSES}; N_VA = {c: USAR[c] - N_AJ[c] for c in CLASSES}
assert all(N_AJ[c] >= 1 and N_VA[c] >= 1 for c in CLASSES), "alguma classe ficaria sem linhas no ajuste ou na validação"
plano = pd.DataFrame({"código": [COD[c] for c in CLASSES], "linhas no treino": [N_TREINO[c] for c in CLASSES], "usadas": [USAR[c] for c in CLASSES], "ajuste": [N_AJ[c] for c in CLASSES], "validação": [N_VA[c] for c in CLASSES],
                      "não usadas": [N_TREINO[c] - USAR[c] for c in CLASSES]}, index=CLASSES)
print(f"{len(CLASSES)} classes (do notebook 03m): {CLASSES}"); display(plano)
print(f"\nPlano: ajuste = {sum(N_AJ.values()):,} linhas | validação = {sum(N_VA.values()):,} linhas | total usado = {sum(USAR.values()):,} (todas reais; nenhuma linha sintética)")

7 classes (do notebook 03m): ['BENIGN', 'LDAP', 'MSSQL', 'NetBIOS', 'Syn', 'UDP', 'UDPLag']


,código,linhas no treino,usadas,ajuste,validação,não usadas
BENIGN,0,50768,50000,40000,10000,768
LDAP,1,30099,30099,24079,6020,0
MSSQL,2,207003,50000,40000,10000,157003
NetBIOS,3,19490,19490,15592,3898,0
Syn,4,155515,50000,40000,10000,105515
UDP,5,1075337,50000,40000,10000,1025337
UDPLag,6,89066,50000,40000,10000,39066



Plano: ajuste = 239,671 linhas | validação = 59,918 linhas | total usado = 299,589 (todas reais; nenhuma linha sintética)


## 2. A seleção (passo a passo)

1. **Sorteio dentro da classe.** Em cada classe, as linhas são ordenadas por um **número pseudoaleatório fixo** (`cityHash64(id_linha, SEMENTE)`). As primeiras `FRACAO_AJUSTE` das linhas usadas vão para o **ajuste** e as seguintes para a **validação**. Assim os dois grupos **nunca se repetem**, e cada um é um sorteio aleatório da classe.
2. **Quantas linhas por classe.** `min(TETO_POR_CLASSE, linhas da classe)`. Classes pequenas entram inteiras; as grandes são sorteadas até o teto.
3. **Pesos da validação.** Cada linha da validação representa `(linhas da classe no treino) ÷ (linhas da classe na validação)` linhas do treino. A soma dos pesos de cada classe é, portanto, o tamanho dela no treino, e a validação ponderada reproduz as **proporções do treino das 7 classes**. Isso importa para precisão, F1 e PR-AUC; recall e taxa de falso alarme por classe não dependem da proporção.

Para não ordenar milhões de linhas num servidor com pouca memória, o notebook primeiro pega, em cada classe grande, um subconjunto aleatório um pouco maior que o necessário (filtro por `hash % 1.000.000`), e só ordena esse subconjunto.

In [6]:
# ---------- 2.1 Seleção ----------
def selecionar():
    linhas, M, margem0 = [], 1_000_000, 1.25
    for classe in CLASSES:
        N, n_usar = N_TREINO[classe], USAR[classe]; rot = f"({lit(classe)}, {lit('DrDoS_' + classe)})"; margem = margem0
        for _ in range(4):
            K = M if N <= n_usar else min(M, max(1, math.ceil(M * min(1.0, (margem * n_usar + 6 * math.sqrt(n_usar)) / N))))
            onde_h = "" if K >= M else f" AND cityHash64(id_linha, {SEMENTE}) % {M} < {K}"
            sub = consultar(f"SELECT toString(id_linha) AS id, toString(cityHash64(id_linha, {SEMENTE})) AS h FROM prep_final_treino WHERE label_multi IN {rot}{onde_h}")
            if len(sub) >= n_usar: break
            margem *= 1.6
        else:
            raise RuntimeError(f"o subconjunto aleatório de {classe} ficou menor que o necessário; aumente a margem")
        sub = sub.assign(hh=sub.h.map(int)).sort_values("hh", kind="stable").reset_index(drop=True).head(n_usar)
        peso_val = N / N_VA[classe]
        for k, i in enumerate(sub.id): linhas.append((i, "treino" if k < N_AJ[classe] else "validacao", COD[classe], classe, 1.0 if k < N_AJ[classe] else peso_val))
    return pd.DataFrame(linhas, columns=["id", "papel", "y", "tipo", "peso"])

SEL = selecionar()
assert SEL.id.is_unique and SEL.id.map(str.isdigit).all(), "ids repetidos ou inválidos"
print(f"✔ Selecionadas {len(SEL):,} linhas: ajuste = {int((SEL.papel == 'treino').sum()):,} | validação = {int((SEL.papel == 'validacao').sum()):,} (com pesos)")
print(SEL.groupby(["tipo", "papel"]).size().unstack(fill_value=0).reindex(CLASSES).to_string())

✔ Selecionadas 299,589 linhas: ajuste = 239,671 | validação = 59,918 (com pesos)
papel    treino  validacao
tipo                      
BENIGN    40000      10000
LDAP      24079       6020
MSSQL     40000      10000
NetBIOS   15592       3898
Syn       40000      10000
UDP       40000      10000
UDPLag    40000      10000


## 3. Gravação: auditoria, dados e views

Grava a seleção em `prep_bal_multi_selecao`, copia os dados dessas linhas da view definitiva para `prep_bal_multi_dados` (uma tabela pequena, com a seleção congelada) e cria as duas views usadas pelos notebooks de treino. A classe por nome (`tipo`) e o papel ficam **fora** das views.

In [7]:
# ---------- 3.1 Gravação ----------
versao_bal = datetime.now().strftime("%Y%m%d_%H%M%S")
executar(f"DROP TABLE IF EXISTS `{SEL_TAB}`")
executar(f"CREATE TABLE {SEL_TAB} (id_linha UInt64, papel String, y UInt8, tipo String, peso Float64, versao String) ENGINE = MergeTree ORDER BY (papel, id_linha)")
for i in range(0, len(SEL), 20_000):
    pedaco = SEL.iloc[i:i + 20_000]
    executar_robusto(f"INSERT INTO {SEL_TAB} VALUES " + ", ".join(f"({r.id}, {lit(r.papel)}, {int(r.y)}, {lit(r.tipo)}, {float(r.peso)!r}, {lit(versao_bal)})" for r in pedaco.itertuples()))
print(f"✔ {SEL_TAB}: {n_linhas(SEL_TAB):,} linhas")

executar_robusto(f"CREATE TABLE {DADOS_TAB} ENGINE = MergeTree ORDER BY (papel, id_linha) AS "
                 f"SELECT s.papel AS papel, s.tipo AS tipo, s.peso AS peso, v.* FROM {VIEW_BASE} v INNER JOIN {SEL_TAB} s ON v.id_linha = s.id_linha SETTINGS max_threads = 2",
                 antes=f"DROP TABLE IF EXISTS `{DADOS_TAB}`")
print(f"✔ {DADOS_TAB}: {n_linhas(DADOS_TAB):,} linhas")

cols_base = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(VIEW_BASE)} ORDER BY position")["name"].tolist()
cols_modelo = ", ".join(q(c) for c in cols_base if c not in ("id_linha", "y"))
executar("DROP VIEW IF EXISTS `prep_vw_bal_multi_refit`")        # o refit (treino final com tudo) fica para o notebook de treino
executar(f"DROP VIEW IF EXISTS `{V_TREINO}`")
executar(f"CREATE VIEW {V_TREINO} AS SELECT id_linha, y, {cols_modelo} FROM {DADOS_TAB} WHERE papel = 'treino'")
executar(f"DROP VIEW IF EXISTS `{V_VALID}`")
executar(f"CREATE VIEW {V_VALID} AS SELECT id_linha, y, peso, {cols_modelo} FROM {DADOS_TAB} WHERE papel = 'validacao'")
print(f"✔ Views: {V_TREINO} e {V_VALID}  | versão {versao_bal}")

✔ prep_bal_multi_selecao: 299,589 linhas
✔ prep_bal_multi_dados: 299,589 linhas
✔ Views: prep_vw_bal_multi_treino e prep_vw_bal_multi_validacao  | versão 20261027_173009


## 4. Validação

Confere de forma independente: as quantidades por classe e papel, que cada classe usa o teto (ou todas as suas linhas), que ajuste e validação são **disjuntos**, que as linhas existem no treino com a classe certa, que os **pesos** da validação somam o tamanho de cada classe no treino, que os dados copiados são **idênticos** aos da view definitiva e que as views têm as colunas certas, sem `tipo`, `papel` nem rótulos, e sem nulos.

In [8]:
# ---------- 4.1 Validação ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

d = consultar_num(f"SELECT papel, tipo, count() AS n, sum(peso) AS peso FROM {SEL_TAB} GROUP BY papel, tipo")
g = {(r.papel, r.tipo): (int(r.n), float(r.peso)) for r in d.itertuples()}
verifica("Cada classe tem no ajuste e na validação as quantidades planejadas", all(g.get(("treino", c), (0, 0))[0] == N_AJ[c] and g.get(("validacao", c), (0, 0))[0] == N_VA[c] for c in CLASSES), f"({len(CLASSES)} classes)")
verifica("Cada classe usa o teto de linhas, ou todas as suas linhas quando tem menos que o teto", all(N_AJ[c] + N_VA[c] == min(TETO_POR_CLASSE, N_TREINO[c]) for c in CLASSES))
verifica("Divisão de cada classe = FRACAO_AJUSTE das linhas usadas", all(N_AJ[c] == int(FRACAO_AJUSTE * (N_AJ[c] + N_VA[c])) for c in CLASSES), f"({FRACAO_AJUSTE:.0%} no ajuste)")
verifica("Papéis disjuntos: nenhuma linha em treino e validação ao mesmo tempo", n_linhas(SEL_TAB) == int(consultar_num(f"SELECT uniqExact(id_linha) AS n FROM {SEL_TAB}").n[0]) == len(SEL))
verifica("Todas as linhas escolhidas existem em prep_final_treino com a classe certa",
         int(consultar_num(f"SELECT count() AS n FROM {SEL_TAB} s INNER JOIN prep_final_treino f ON s.id_linha = f.id_linha WHERE replaceRegexpOne(toString(f.label_multi), '^DrDoS_', '') = s.tipo").n[0]) == len(SEL))
verifica("Pesos da validação somam as linhas de cada classe no treino", all(abs(g[("validacao", c)][1] - N_TREINO[c]) < 1e-6 * N_TREINO[c] for c in CLASSES))
dd = consultar_num(f"SELECT y, tipo, count() AS n FROM {DADOS_TAB} GROUP BY y, tipo")
verifica("y coerente com a classe (código de prep_classes)", all(COD[r.tipo] == int(r.y) for r in dd.itertuples()))
verifica("Tabela de dados = tabela de seleção (mesmo número de linhas)", n_linhas(DADOS_TAB) == len(SEL), f"({n_linhas(DADOS_TAB):,})")
verifica("Todas as classes aparecem no ajuste e na validação", all(("treino", c) in g and ("validacao", c) in g for c in CLASSES))

feats = [c for c in cols_base if c not in ("id_linha", "y")]
v = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {DADOS_TAB} WHERE cityHash64(id_linha) % 20 = 0 ORDER BY id")
w = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in feats) + f" FROM {VIEW_BASE} WHERE id_linha IN (SELECT id_linha FROM {DADOS_TAB} WHERE cityHash64(id_linha) % 20 = 0) ORDER BY id")
igual = len(v) == len(w) and (v["id"].values == w["id"].values).all() and np.allclose(v[["y"] + feats].apply(pd.to_numeric).values.astype(float), w[["y"] + feats].apply(pd.to_numeric).values.astype(float), rtol=1e-12, atol=1e-12)
verifica("Dados idênticos aos da view definitiva (amostra)", igual and len(v) > 0, f"({len(v):,} linhas comparadas)")

proibidas = {"tipo", "papel", "label_original", "label_multi", "label_binary", "origem_tabela", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "b"}
for vw, esp in ((V_TREINO, ["id_linha", "y"] + feats), (V_VALID, ["id_linha", "y", "peso"] + feats)):
    cv = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(vw)} ORDER BY position")["name"].tolist()
    verifica(f"{vw}: colunas esperadas e sem tipo/papel/rótulos", cv == esp and not (proibidas & set(cv)), f"({len(cv)} colunas)")
    nulos = consultar_num(f"SELECT " + ", ".join(f"countIf(isNull({q(c)})) AS {q(c)}" for c in feats) + f" FROM {vw}").iloc[0]
    verifica(f"{vw}: sem nulos nos atributos", int(nulos.sum()) == 0)
verifica("Teste intocado: a view de teste continua com as linhas do notebook 03m", n_linhas(VIEW_TESTE) == sum(N_TESTE.values()), f"({sum(N_TESTE.values()):,})")
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")

✔ Cada classe tem no ajuste e na validação as quantidades planejadas (7 classes)
✔ Cada classe usa o teto de linhas, ou todas as suas linhas quando tem menos que o teto 
✔ Divisão de cada classe = FRACAO_AJUSTE das linhas usadas (80% no ajuste)
✔ Papéis disjuntos: nenhuma linha em treino e validação ao mesmo tempo 
✔ Todas as linhas escolhidas existem em prep_final_treino com a classe certa 
✔ Pesos da validação somam as linhas de cada classe no treino 
✔ y coerente com a classe (código de prep_classes) 
✔ Tabela de dados = tabela de seleção (mesmo número de linhas) (299,589)
✔ Todas as classes aparecem no ajuste e na validação 
✔ Dados idênticos aos da view definitiva (amostra) (14,932 linhas comparadas)
✔ prep_vw_bal_multi_treino: colunas esperadas e sem tipo/papel/rótulos (46 colunas)
✔ prep_vw_bal_multi_treino: sem nulos nos atributos 
✔ prep_vw_bal_multi_validacao: colunas esperadas e sem tipo/papel/rótulos (47 colunas)
✔ prep_vw_bal_multi_validacao: sem nulos nos atributos 
✔ Tes

## 5. Resumo e leitura

Mostra, por classe, quanto há no treino, quanto foi usado e como se divide, o **peso de cada linha da validação** e o **peso `class_weight="balanced"`** que o RF vai usar no notebook 05m (calculado só com o ajuste: `linhas do ajuste ÷ (nº de classes × linhas da classe no ajuste)`). Mostra também a composição do teste, que **não** é balanceado. A função `ler_bal_multi` lê o ajuste e a validação (conjuntos pequenos: leitura inteira).

In [9]:
/[# ---------- 5.1 Resumo e leitura ----------
r = consultar_num(f"SELECT tipo, papel, count() AS n, avg(peso) AS peso FROM {SEL_TAB} GROUP BY tipo, papel")
t = r.pivot_table(index="tipo", columns="papel", values="n", aggfunc="sum", fill_value=0).astype(int).reindex(CLASSES)
t["no treino (total)"] = pd.Series(N_TREINO); t["peso por linha na validação"] = r[r.papel == "validacao"].set_index("tipo")["peso"].round(2)
n_aj_total = int(t["treino"].sum()); t["peso balanced (ajuste)"] = (n_aj_total / (len(CLASSES) * t["treino"])).round(3)
t = t.rename(columns={"treino": "ajuste", "validacao": "validação (linhas)"})[["no treino (total)", "ajuste", "validação (linhas)", "peso por linha na validação", "peso balanced (ajuste)"]]
print("== Linhas por classe =="); display(t)
print(f"\nRazão entre a maior e a menor classe no ajuste: {t['ajuste'].max() / t['ajuste'].min():.2f} para 1 (o peso `balanced` da menor classe é {t['peso balanced (ajuste)'].max() / t['peso balanced (ajuste)'].min():.2f}× o da maior)")
te = pd.DataFrame({"linhas no teste (sem sobreposição)": pd.Series(N_TESTE)}).reindex(CLASSES); te["% do teste"] = (100 * te.iloc[:, 0] / te.iloc[:, 0].sum()).round(3)
print("\nTeste (NÃO balanceado, como está):"); display(te)
cols_fin = set(consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_final_treino'")["name"])
col_conf = next((c for c in ("conflito_rotulo", "conflito_binario") if c in cols_fin), None)
if col_conf:
    cf = n_linhas("prep_final_treino", f"{col_conf} = 1 AND id_linha IN (SELECT id_linha FROM {SEL_TAB} WHERE papel = 'treino')")
    print(f"Linhas com marca de conflito de rótulo ({col_conf}) dentro do ajuste: {cf:,} (apenas contadas; nenhuma foi removida)")
else: print("(prep_final_treino não tem coluna de conflito de rótulo: contagem de conflitos omitida)")
print(f"RAM do ajuste em float64: {n_linhas(DADOS_TAB, 'papel = ' + lit('treino')) * len(feats) * 8 / 2**20:.1f} MiB")

def ler_bal_multi(papel="treino"):
    # papel: 'treino' (ajuste) ou 'validacao' (com a coluna peso). Os conjuntos são pequenos: leitura inteira.
    return consultar_num(f"SELECT * FROM {V_TREINO if papel == 'treino' else V_VALID}")

for p in ("treino", "validacao"):
    a = ler_bal_multi(p); assert not a.drop(columns=["id_linha"]).isna().any().any()
    print(f"Leitura de teste ({p}): {a.shape[0]:,} linhas x {a.shape[1]} colunas, sem NaN | classes: { {CLASSES[int(k)]: int(v) for k, v in a['y'].value_counts().sort_index().items()} }")

== Linhas por classe ==


papel,no treino (total),ajuste,validação (linhas),peso por linha na validação,peso balanced (ajuste)
tipo,,,,,
BENIGN,50768,40000,10000,5.08,0.856
LDAP,30099,24079,6020,5.00,1.422
MSSQL,207003,40000,10000,20.70,0.856
NetBIOS,19490,15592,3898,5.00,2.196
Syn,155515,40000,10000,15.55,0.856
UDP,1075337,40000,10000,107.53,0.856
UDPLag,89066,40000,10000,8.91,0.856



Razão entre a maior e a menor classe no ajuste: 2.57 para 1 (o peso `balanced` da menor classe é 2.57× o da maior)

Teste (NÃO balanceado, como está):


,linhas no teste (sem sobreposição),% do teste
BENIGN,56216,0.779
LDAP,112491,1.559
MSSQL,854242,11.837
NetBIOS,91175,1.263
Syn,4887060,67.721
UDP,1214746,16.833
UDPLag,557,0.008


Linhas com marca de conflito de rótulo (conflito_rotulo) dentro do ajuste: 33,322 (apenas contadas; nenhuma foi removida)
RAM do ajuste em float64: 80.5 MiB
Leitura de teste (treino): 239,671 linhas x 46 colunas, sem NaN | classes: {'BENIGN': 40000, 'LDAP': 24079, 'MSSQL': 40000, 'NetBIOS': 15592, 'Syn': 40000, 'UDP': 40000, 'UDPLag': 40000}
Leitura de teste (validacao): 59,918 linhas x 47 colunas, sem NaN | classes: {'BENIGN': 10000, 'LDAP': 6020, 'MSSQL': 10000, 'NetBIOS': 3898, 'Syn': 10000, 'UDP': 10000, 'UDPLag': 10000}


In [10]:
a = consultar_num(f"SELECT t.tipo AS classe, count() AS linhas, countIf(f.conflito_rotulo = 1) AS em_conflito FROM {SEL_TAB} t INNER JOIN prep_final_treino f ON t.id_linha = f.id_linha WHERE t.papel = 'treino' GROUP BY t.tipo ORDER BY t.tipo")
a["% em conflito"] = (100 * a.em_conflito / a.linhas).round(1); print(a.to_string(index=False)); print()
b = consultar_num("SELECT arraySort(rotulos) AS rotulos_em_conflito, count() AS grupos, sum(n_linhas) AS linhas_originais FROM prep_grp_treino WHERE n_rotulos > 1 GROUP BY rotulos_em_conflito ORDER BY grupos DESC LIMIT 25"); print(b.to_string(index=False))

  conexão instável (ConnectionAbortedError: [WinError 10053] Uma conexão estabelecida foi anulada pelo software no computador host); nova tentativa em 5s...
 classe  linhas  em_conflito  % em conflito
 BENIGN   40000           17            0.0
   LDAP   24079        10323           42.9
  MSSQL   40000         3784            9.5
NetBIOS   15592         8149           52.3
    Syn   40000          647            1.6
    UDP   40000         5559           13.9
 UDPLag   40000         4843           12.1

                                           rotulos_em_conflito  grupos  linhas_originais
                                       [DrDoS_SSDP, DrDoS_UDP]  136809           1212660
                                      [DrDoS_DNS, DrDoS_MSSQL]    6002            328634
                               [DrDoS_SSDP, DrDoS_UDP, UDPLag]    5198            560648
                                       [DrDoS_DNS, DrDoS_LDAP]    5101            159117
                                       [DrDoS